# Master control panel

Builds `pruefstellen_master` from the per-source exports. All logic lives in `core.master_*`; the cells only call it.
Order: 1 config and inventory, 2 dev run, 3 summary and rows to review, 4 spot-check one master_id, 5 change report, 6 official run.
The only network use is OpenStreetMap Nominatim for rows without usable coordinates (cap 60 requests, cached in `4_reference/geocode_cache/`).
Personal data is in the outputs: keep `3_master_output/` out of git and out of chat.

## 1. Config and inventory

In [ ]:
import pandas as pd
from core import master_inputs as mi, master_run, master_schema as ms, paths
pd.set_option('display.width', 220, 'display.max_colwidth', 70, 'display.max_rows', 200)

LIVE_GEOCODING = False  # True = ask Nominatim (cap 60, cached); False = cache only, no request
print('INCLUDE_OUT_OF_SCOPE =', ms.INCLUDE_OUT_OF_SCOPE, '| MERGE_ACROSS_UO =', ms.MERGE_ACROSS_UO, '| pins =', mi.PINS)
inv = pd.DataFrame(mi.inventory())
inv[['slug', 'run', 'mode', 'why', 'rows', 'in_scope', 'status_count', 'matches_status']]

## 2. Dev run (writes `3_master_output/_dev/<date_time>/`)

In [ ]:
res = master_run.run('dev', live_geocoding=LIVE_GEOCODING)
print(res.status, '->', res.out_dir)

## 3. Summary and rows to review

In [ ]:
print(master_run.render_md(res.summary))

In [ ]:
m = res.master
m[m['needs_review'] == 'True'][['master_id', 'uo', 'listed_by', 'street', 'zip', 'city', 'needs_review_reason', 'doubt_flags']]

## 4. Spot-check one master row (all contributing source rows side by side)

In [ ]:
MASTER_ID = 'PS-000001'
master_run.spot_check(MASTER_ID, res.out_dir)

## 5. Change report against the latest official master

In [ ]:
res.diff

## 6. Official run
Writes `3_master_output/YYYY-MM/` and refuses if that folder exists. Only after the dev run passed and the rows to review were looked at.

In [ ]:
CONFIRM_OFFICIAL = False
if CONFIRM_OFFICIAL:
    official = master_run.run('official', confirm=True, live_geocoding=LIVE_GEOCODING)
    print(official.status, '->', official.out_dir)
else:
    print('official run not started (CONFIRM_OFFICIAL is False)')